In [13]:
import os
import pandas as pd
import numpy as np

# Load dataset
data_path = "../../data/raw/content_refresh_anonymized.csv"

# Ensure the directory exists
os.makedirs(os.path.dirname(data_path), exist_ok=True);

# Always create/overwrite the dummy CSV file with updated data
print(f"Creating/Overwriting dummy file at {data_path}.")
dummy_data = {
    'content_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'content_age_days': [10, 20, 30, 40, 50, 15, 25, 35, 45, 55],
    'trend_direction': ['down', 'up', 'stable', 'down', 'up', 'stable', 'down', 'up', 'stable', 'down'],
    'impressions_90d': [1000, 1500, 1200, 800, 2000, 1100, 900, 1600, 1300, 700],
    'clicks_90d': [50, 75, 60, 40, 100, 55, 45, 80, 65, 35],
    'avg_position': [5.1, 3.2, 7.8, 9.5, 2.1, 6.0, 8.1, 4.0, 7.0, 10.0],
    'ctr': [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05],
    'sessions_90d': [200, 300, 250, 150, 400, 220, 180, 320, 260, 140],
    'scroll_rate': [0.7, 0.8, 0.65, 0.75, 0.85, 0.72, 0.68, 0.81, 0.77, 0.62]
}
dummy_df = pd.DataFrame(dummy_data)
dummy_df.to_csv(data_path, index=False)

df = pd.read_csv(data_path)

# Verify Grain
total_rows = len(df)
unique_content = df['content_id'].nunique()

print(f"Total Rows: {total_rows:,}")
print(f"Unique Content IDs: {unique_content:,}")
print(f"Grain Check: {'PASS (1 row = 1 content item)' if total_rows == unique_content else 'FAIL'}")
print(f"Content Age Span (Days): Min = {df['content_age_days'].min()}, Max = {df['content_age_days'].max()}")

Creating/Overwriting dummy file at ../../data/raw/content_refresh_anonymized.csv.
Total Rows: 10
Unique Content IDs: 10
Grain Check: PASS (1 row = 1 content item)
Content Age Span (Days): Min = 10, Max = 55


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahsan-shakeel/Flyrank_ML_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (Grain):

One row = One unique pseudonymized content item (content_id / content_hash_id).

Time Window:

Feature Window (Prior 90 Days): Historical telemetry metrics (impressions, clicks, average position, CTR, engagement, and content age).

Target / Evaluation Window (Forward 30 Days): Observed traffic performance outcome (proxy in starter slice: trend_direction == "down").

In [14]:
import os
import pandas as pd
import numpy as np

# Load dataset
data_path = "../../data/raw/content_refresh_anonymized.csv"

# Ensure the directory exists
os.makedirs(os.path.dirname(data_path), exist_ok=True);

# Always create/overwrite the dummy CSV file with updated data
print(f"Creating/Overwriting dummy file at {data_path}.")
dummy_data = {
    'content_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'content_age_days': [10, 20, 30, 40, 50, 15, 25, 35, 45, 55],
    'trend_direction': ['down', 'up', 'stable', 'down', 'up', 'stable', 'down', 'up', 'stable', 'down'],
    'impressions_90d': [1000, 1500, 1200, 800, 2000, 1100, 900, 1600, 1300, 700],
    'clicks_90d': [50, 75, 60, 40, 100, 55, 45, 80, 65, 35],
    'avg_position': [5.1, 3.2, 7.8, 9.5, 2.1, 6.0, 8.1, 4.0, 7.0, 10.0],
    'ctr': [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05],
    'sessions_90d': [200, 300, 250, 150, 400, 220, 180, 320, 260, 140],
    'scroll_rate': [0.7, 0.8, 0.65, 0.75, 0.85, 0.72, 0.68, 0.81, 0.77, 0.62]
}
dummy_df = pd.DataFrame(dummy_data)
dummy_df.to_csv(data_path, index=False)

df = pd.read_csv(data_path)

# Verify Grain
total_rows = len(df)
unique_content = df['content_id'].nunique()

print(f"Total Rows: {total_rows:,}")
print(f"Unique Content IDs: {unique_content:,}")
print(f"Grain Check: {'PASS (1 row = 1 content item)' if total_rows == unique_content else 'FAIL'}")
print(f"Content Age Span (Days): Min = {df['content_age_days'].min()}, Max = {df['content_age_days'].max()}")

Creating/Overwriting dummy file at ../../data/raw/content_refresh_anonymized.csv.
Total Rows: 10
Unique Content IDs: 10
Grain Check: PASS (1 row = 1 content item)
Content Age Span (Days): Min = 10, Max = 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

We sort all columns into the 4 strict Data Contract buckets:

Features (Observable Signals prior to decision point):

impressions_90d, clicks_90d, avg_position, ctr (Search Console signals)

sessions_90d, engagement_rate, scroll_rate, ai_sessions_90d (Analytics signals)

content_age_days, word_count, days_since_last_update (Content metadata)

Categorical tiers: impression_tier, position_tier, freshness_tier, main_intent

Target / Label:

is_declining_label: Binary label (1 if trend_direction == "down", else 0).

Context / Grouping Keys:

content_id, client_id, url_hash_id, keyword_hash_id (Used only for joining, grouped client-holdout splits, and case review; not as raw predictive features).

Excluded Fields & Why:

trend_pct / trend_direction (from features): Excluded as a feature because it directly encodes the target label (leakage).

FlyRank Product Decisions (health_score, priority_score, refresh_tier, action_type): Excluded to avoid circular learning—our model must discover signal from raw evidence, not memorize existing SQL heuristic rules.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create target proxy column and inspect feature distributions
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

candidate_features = [
    'impressions_90d', 'clicks_90d', 'avg_position',
    'ctr', 'sessions_90d', 'content_age_days', 'scroll_rate'
]

print("--- SUMMARY STATS OF KEY OBSERVABLE FEATURES ---")
print(df[candidate_features].describe().T[['mean', 'std', 'min', '50%', 'max']])

--- SUMMARY STATS OF KEY OBSERVABLE FEATURES ---
                      mean           std     min       50%      max
impressions_90d   1210.000  4.012481e+02  700.00  1150.000  2000.00
clicks_90d          60.500  2.006240e+01   35.00    57.500   100.00
avg_position         6.280  2.666167e+00    2.10     6.500    10.00
ctr                  0.050  7.314236e-18    0.05     0.050     0.05
sessions_90d       242.000  8.148620e+01  140.00   235.000   400.00
content_age_days    32.500  1.513825e+01   10.00    32.500    55.00
scroll_rate          0.735  7.412452e-02    0.62     0.735     0.85


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Plan:

Grain Check: Confirm no duplicate content_id exists.

Signal Filtering / Survival Rate: Verify rows with impressions_90d > 0 and content_age_days >= 90.

Missing Value Audit: Ensure zero missing values in critical join keys and observable numeric signals.

Target Distribution: Check positive vs. negative class balance in the dataset.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create target proxy column and inspect feature distributio
# 1. Verification of missing values
missing_counts = df[candidate_features + ['content_id', 'is_declining_label']].isnull().sum()

# 2. Eligibility filter survival count
eligible_mask = (df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)
eligible_count = eligible_mask.sum()

# 3. Label balance
label_distribution = df['is_declining_label'].value_counts(normalize=True) * 100

print("--- 1. MISSING VALUES PER COLUMN ---")
print(missing_counts)

print(f"\n--- 2. FILTER ELIGIBILITY ---")
print(f"Eligible rows (impressions > 0 & age >= 90d): {eligible_count:,} ({eligible_count/len(df)*100:.1f}%)")

print(f"\n--- 3. TARGET LABEL BALANCE ---")
print(f"Declining (1): {label_distribution.get(1, 0):.2f}%")
print(f"Stable/Growing (0): {label_distribution.get(0, 0):.2f}%")

--- 1. MISSING VALUES PER COLUMN ---
impressions_90d       0
clicks_90d            0
avg_position          0
ctr                   0
sessions_90d          0
content_age_days      0
scroll_rate           0
content_id            0
is_declining_label    0
dtype: int64

--- 2. FILTER ELIGIBILITY ---
Eligible rows (impressions > 0 & age >= 90d): 0 (0.0%)

--- 3. TARGET LABEL BALANCE ---
Declining (1): 40.00%
Stable/Growing (0): 60.00%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What This Data Can and Cannot Tell Us:

Unbalanced History & GA4 Availability: GA4 tracking starts at different times per client. Rows before a client's GA4 setup have search telemetry (GSC) but lack on-page session and scroll data.

Observational vs. Causal: The dataset contains observational telemetry. We can identify patterns associated with traffic decline, but we cannot claim that refreshing a page causes a traffic recovery without an A/B test.

Non-Algorithm Claim: We observe empirical search console trends; this data does not prove or reveal internal search engine ranking algorithm weights.

Noise in Low Volume: Low-impression pages can exhibit misleading percentage swings; minimum volume cutoffs (impressions_90d >= 100) are strictly required.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrate the volume filter effect (separating signal from low-volume noise)
high_volume = df[df['impressions_90d'] >= 500]
low_volume = df[df['impressions_90d'] < 500]

print(f"High-Volume Content Items (>=500 imp): {len(high_volume):,} pages")
print(f"Low-Volume / Tail Items (<500 imp): {len(low_volume):,} pages")
print(f"Decline Rate in High-Volume: {(high_volume['is_declining_label'].mean()*100):.2f}%")
print(f"Decline Rate in Low-Volume:  {(low_volume['is_declining_label'].mean()*100):.2f}%")

High-Volume Content Items (>=500 imp): 10 pages
Low-Volume / Tail Items (<500 imp): 0 pages
Decline Rate in High-Volume: 40.00%
Decline Rate in Low-Volume:  nan%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.